In [1]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("../retailiq.db")

# Base: RFM table already has recency, frequency, monetary per customer
rfm = pd.read_sql("SELECT * FROM rfm_segments", conn)

# Repeat purchase behavior
intervals = pd.read_sql("SELECT * FROM repeat_purchase_intervals", conn)

# Review behavior per customer
review_query = """
SELECT c.customer_unique_id,
       AVG(r.review_score) AS avg_review_score,
       COUNT(r.review_id) AS review_count
FROM orders_clean o
JOIN customers_clean c ON o.customer_id = c.customer_id
JOIN reviews_clean r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
"""
reviews_per_cust = pd.read_sql(review_query, conn)

# Delivery experience per customer
delivery_query = """
SELECT c.customer_unique_id,
       AVG(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_purchase_timestamp)) AS avg_delivery_days,
       AVG(JULIANDAY(o.order_delivered_customer_date) - JULIANDAY(o.order_estimated_delivery_date)) AS avg_delay_days
FROM orders_clean o
JOIN customers_clean c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_unique_id
"""
delivery_per_cust = pd.read_sql(delivery_query, conn)

# Geography (for later fairness audit — keep it, don't feed state directly into the model as a predictive feature)
geo_query = """
SELECT DISTINCT c.customer_unique_id, c.customer_state
FROM customers_clean c
"""
geo_per_cust = pd.read_sql(geo_query, conn)

print("rfm:", rfm.shape)
print("intervals:", intervals.shape)
print("reviews_per_cust:", reviews_per_cust.shape)
print("delivery_per_cust:", delivery_per_cust.shape)
print("geo_per_cust:", geo_per_cust.shape)

rfm: (93357, 9)
intervals: (2801, 5)
reviews_per_cust: (92755, 3)
delivery_per_cust: (93350, 3)
geo_per_cust: (96136, 2)


In [2]:
features = rfm.merge(intervals, on="customer_unique_id", how="left")
features = features.merge(reviews_per_cust, on="customer_unique_id", how="left")
features = features.merge(delivery_per_cust, on="customer_unique_id", how="left")
features = features.merge(geo_per_cust, on="customer_unique_id", how="left")

# Customers with no repeat orders won't have interval data -- fill sensibly
features["repeat_order_count"] = features["repeat_order_count"].fillna(0)
features["avg_days_between_orders"] = features["avg_days_between_orders"].fillna(-1)  # -1 = never repeated, a valid signal not a missing value

print(features.shape)
features.head()
print("\nNulls remaining:\n", features.isnull().sum())

(93397, 18)

Nulls remaining:
 customer_unique_id             0
recency_days                   0
frequency                      0
monetary                       0
r_score                        0
f_score                        0
m_score                        0
rfm_total                      0
segment                        0
repeat_order_count             0
avg_days_between_orders        0
min_days_between_orders    90558
max_days_between_orders    90558
avg_review_score             603
review_count                 603
avg_delivery_days              8
avg_delay_days                 8
customer_state                 0
dtype: int64


In [3]:
# Churn definition: no purchase in the 90 days following their last order,
# relative to the dataset's most recent date. Standard e-commerce churn window.
max_date_query = "SELECT MAX(order_purchase_timestamp) AS ref_date FROM orders_clean"
ref_date = pd.read_sql(max_date_query, conn)["ref_date"][0]
ref_date = pd.to_datetime(ref_date)

features["churned"] = (features["recency_days"] > 90).astype(int)

print(features["churned"].value_counts(normalize=True))

churned
1    0.801568
0    0.198432
Name: proportion, dtype: float64


In [4]:
features.to_sql("customer_features", conn, if_exists="replace", index=False)
conn.close()
print("Saved as 'customer_features' table")

Saved as 'customer_features' table
